# Module 20 — Production Tokenizers

Module 19 built BPE from scratch and it genuinely worked — but it was
trained on ~100 words with 40 merges, in pure Python. Real tokenizers use
the same core algorithm at a completely different scale: trained on
billions of characters, tens-of-thousands of merges, implemented in
Rust/C++ so they can tokenize gigabytes of text per second. This module
swaps in two real libraries: **`tiktoken`** (OpenAI's GPT-2/GPT-4
tokenizer) and Hugging Face's **`tokenizers`** (for training a fresh BPE
vocabulary, the production version of what Module 19 did by hand).

## 1. `tiktoken`: GPT-2's pretrained tokenizer

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
print(f"GPT-2 vocab size: {enc.n_vocab:,}")

sample = "the traveler and paimon explore mondstadt"
gpt2_tokens = enc.encode(sample)
print(f"tokens: {gpt2_tokens}")
print(f"token count: {len(gpt2_tokens)}")
print("decoded pieces:", [enc.decode([t]) for t in gpt2_tokens])

decoded = enc.decode(gpt2_tokens)
assert decoded == sample
print(f"\nRound-trip confirmed: decode(encode(text)) == text.")

## 2. Common words vs. proper nouns GPT-2 never trained on

GPT-2's tokenizer was trained on a huge, mostly-English web corpus that
predates Genshin Impact. Common English words should compress to a single
token; game-specific proper nouns like "Mondstadt" or "Paimon" should
fragment into several smaller pieces — a direct, real-world example of
Module 19's "rare words cost more tokens" idea.

In [ ]:
words_to_check = ["the", "traveler", "explore", "paimon", "mondstadt", "genshin", "zhongli"]
for word in words_to_check:
    tokens = enc.encode(word)
    pieces = [enc.decode([t]) for t in tokens]
    print(f"{word!r:>12} -> {len(tokens)} token(s): {pieces}")

## 3. Training your own tokenizer with Hugging Face `tokenizers`

This is the production version of Module 19's from-scratch training loop —
same BPE algorithm, real implementation, trained on our own tiny corpus so
we can compare directly.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

CORPUS = """Aether and Lumine are twins known as the Traveler. They came from another world and lost each other upon arrival in Teyvat. Paimon found Aether floating near Mondstadt and decided to travel together. Mondstadt is called the City of Freedom, and the wind blows gently across its hills. Klee loves to explore the city and often causes small explosions with her bombs. Diluc runs the Dawn Winery outside the city walls. Kaeya works at the Knights of Favonius and enjoys teasing Diluc. Jean leads the Knights of Favonius with great responsibility. Barbara sings songs at the church and heals the sick."""

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=120, special_tokens=["[UNK]"])
tokenizer.train_from_iterator([CORPUS], trainer=trainer)

print(f"trained vocab size: {tokenizer.get_vocab_size()}")

encoded = tokenizer.encode(sample)
print(f"tokens for {sample!r}: {encoded.tokens}")
print(f"token count: {len(encoded.tokens)}")

decoded_hf = tokenizer.decode(encoded.ids)
print(f"decoded: {decoded_hf!r}")

## 4. Comparing all three, on the same sentence

In [ ]:
char_count = len(sample.replace(" ", "_"))
scratch_bpe_count = 26  # Module 19's from-scratch result, same sentence

print(f"Character-level (Modules 06-18):     {char_count} tokens")
print(f"From-scratch BPE (Module 19, 40 merges, ~100-word corpus): {scratch_bpe_count} tokens")
print(f"HF tokenizers BPE (trained on the same tiny corpus):        {len(encoded.tokens)} tokens")
print(f"tiktoken GPT-2 (pretrained on a massive real corpus):       {len(gpt2_tokens)} tokens")

## Recap

- `tiktoken` and Hugging Face `tokenizers` implement the exact same BPE
  idea from Module 19, just trained at real scale and running in
  optimized native code.
- Confirmed concretely: GPT-2's tokenizer round-trips correctly, and
  splits game-specific proper nouns (never in its training data) into
  multiple pieces while common English words stay single tokens — Module
  19's "rare words cost more" idea, shown on real vocabulary.
- Training our own tokenizer with `tokenizers` on the exact same tiny
  corpus as Module 19 gives a direct apples-to-apples comparison against
  the from-scratch version.

Module 30's real pretraining corpus will use one of these production
tokenizers (not the from-scratch Module 19 version) — Module 21 next
covers the data pipeline needed to actually feed a large corpus through a
tokenizer efficiently.